<a href="https://colab.research.google.com/github/abdullahalminhaz/Deep-Learning-Notebooks/blob/main/age_gender_funtional_model/age_gender_funtional_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
#!/bin/bash
!kaggle datasets download jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
 75% 250M/331M [00:00<00:00, 739MB/s] 
100% 331M/331M [00:00<00:00, 751MB/s]


In [ ]:
import zipfile
with zipfile.ZipFile('/content/utkface-new.zip', 'r') as zip_ref:
    zip_ref.extractall()
    zip_ref.close()

In [ ]:
!pip install keras_preprocessing

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.6 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
from keras_preprocessing.image import ImageDataGenerator

In [ ]:
folder_path = '/content/UTKFace'

In [ ]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [ ]:
len(age)

23708

In [ ]:
df = pd.DataFrame({'age':age,'gender':gender,'img_path':img_path})

In [ ]:
df.shape

(23708, 3)

In [ ]:
df.head()

,age,gender,img_path
0,48,1,48_1_1_20170113001756978.jpg.chip.jpg
1,40,0,40_0_4_20170104192730423.jpg.chip.jpg
2,1,1,1_1_1_20170109194716353.jpg.chip.jpg
3,35,1,35_1_0_20170117091849716.jpg.chip.jpg
4,1,1,1_1_2_20161219211316389.jpg.chip.jpg


In [ ]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac =1, random_state=0).iloc[20000:]

In [ ]:
train_df.shape

(20000, 3)

In [ ]:
test_df.shape

(3708, 3)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/UTKFace',
    x_col='img_path',
    y_col=['age', 'gender'],
    target_size=(200, 200),
    class_mode='multi_output'
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='/content/UTKFace',
    x_col='img_path',
    y_col=['age', 'gender'],
    target_size=(200, 200),
    class_mode='multi_output'
)

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [ ]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200, 200, 3))

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200, 200, 3))

resnet.trainable = False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512, activation='relu')(flatten)

dense3 = Dense(512, activation='relu')(dense1)
dense4 = Dense(512, activation='relu')(dense2)

age_output = Dense(1, activation='linear', name='age')(dense3)

gender_output = Dense(1, activation='sigmoid', name='gender')(dense4)

In [ ]:
model = Model(inputs=resnet.input, outputs=[age_output, gender_output])

In [ ]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'})

In [ ]:
model.fit(train_generator, batch_size=32, epochs=10, validation_data=test_generator)

TypeError: `output_signature` must contain objects that are subclass of `tf.TypeSpec` but found <class 'list'> which is not.